# Paso de merge

Une el resumen EEG de theta por sujeto (media + variabilidad a nivel ROI,
task eyes-open, NS vs SD) con las deltas conductuales, por `participant_id`.
Esta es la tabla que alimenta el scatter de vulnerabilidad (Q3, Q5, Q7) --
el topomap (Q1, Q2) y el panel de dinámica intra-grabación (Q6) consumen
directamente los archivos más granulares `eeg_theta_per_electrode.csv` /
`eeg_theta_epochs_roi.csv`.

Correr después de `01_behavioral_pipeline.ipynb` y `02_eeg_pipeline.ipynb`.

In [1]:
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[2]  # deliveries/week06/code -> raíz del repo
PROCESSED_DIR = REPO_ROOT / "deliveries" / "week06" / "data" / "processed"

TASK = "eyesopen"  # presente para los 71 sujetos; eyesclosed solo en un subconjunto de 38
ROIS = ["frontal", "centro_temporal"]

# Debe coincidir exactamente con FRONTAL_ROI / CENTROTEMPORAL_ROI de
# 02_eeg_pipeline.ipynb (duplicado aquí en vez de importado, ya que importar
# entre notebooks es incómodo).
FRONTAL_ROI = [
    "Fp1", "Fp2", "Fpz", "AF3", "AF4", "AF7", "AF8",
    "F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "Fz",
]
CENTROTEMPORAL_ROI = [
    "FC1", "FC2", "FC3", "FC4", "FC5", "FC6", "FT7", "FT8",
    "C1", "C2", "C3", "C4", "C5", "C6", "Cz", "T7", "T8",
    "CP1", "CP2", "CP3", "CP4", "CP5", "CP6", "CPz", "TP7", "TP8", "TP9", "TP10",
]


def roi_for_channel(ch_name: str) -> str:
    if ch_name in FRONTAL_ROI:
        return "frontal"
    if ch_name in CENTROTEMPORAL_ROI:
        return "centro_temporal"
    return "other"

In [2]:
def eeg_subject_condition_roi_summary() -> pd.DataFrame:
    per_electrode = pd.read_csv(PROCESSED_DIR / "eeg_theta_per_electrode.csv")
    per_electrode = per_electrode[per_electrode["task"] == TASK].copy()
    per_electrode["roi"] = per_electrode["electrode"].map(roi_for_channel)
    per_electrode = per_electrode[per_electrode["roi"].isin(ROIS)]

    # Promedia theta_mean (potencia) y theta_std (variabilidad entre épocas)
    # entre los electrodos de cada ROI, por sujeto/condición.
    roi_summary = (
        per_electrode.groupby(["participant_id", "condition", "roi"])
        .agg(theta_power=("theta_mean", "mean"), theta_variability=("theta_std", "mean"))
        .reset_index()
    )

    wide = roi_summary.pivot(index=["participant_id", "condition"], columns="roi", values=["theta_power", "theta_variability"])
    wide.columns = [f"{roi}_{metric}" for metric, roi in wide.columns]
    wide = wide.reset_index()
    return wide


eeg_wide = eeg_subject_condition_roi_summary()
eeg_wide.head()

,participant_id,condition,centro_temporal_theta_power,frontal_theta_power,centro_temporal_theta_variability,frontal_theta_variability
0,sub-01,NS,0.192474,0.268311,0.071979,0.104040
1,sub-02,NS,0.152268,0.188429,0.062251,0.068228
2,sub-02,SD,0.146162,0.152975,0.061214,0.074375
3,sub-03,NS,0.185690,0.172485,0.072548,0.070793
4,sub-03,SD,0.230614,0.221250,0.097844,0.111907


In [3]:
def add_deltas(wide: pd.DataFrame) -> pd.DataFrame:
    ns = wide[wide["condition"] == "NS"].drop(columns="condition")
    sd = wide[wide["condition"] == "SD"].drop(columns="condition")
    merged = ns.merge(sd, on="participant_id", suffixes=("_NS", "_SD"), how="outer")

    value_cols = [c for c in ns.columns if c != "participant_id"]
    for col in value_cols:
        merged[f"delta_{col}"] = merged[f"{col}_SD"] - merged[f"{col}_NS"]

    return merged


eeg_deltas = add_deltas(eeg_wide)
eeg_deltas.head()

,participant_id,centro_temporal_theta_power_NS,frontal_theta_power_NS,centro_temporal_theta_variability_NS,frontal_theta_variability_NS,centro_temporal_theta_power_SD,frontal_theta_power_SD,centro_temporal_theta_variability_SD,frontal_theta_variability_SD,delta_centro_temporal_theta_power,delta_frontal_theta_power,delta_centro_temporal_theta_variability,delta_frontal_theta_variability
0,sub-01,0.192474,0.268311,0.071979,0.104040,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,sub-02,0.152268,0.188429,0.062251,0.068228,0.146162,0.152975,0.061214,0.074375,-0.006106,-0.035454,-0.001036,0.006148
2,sub-03,0.185690,0.172485,0.072548,0.070793,0.230614,0.221250,0.097844,0.111907,0.044924,0.048764,0.025296,0.041114
3,sub-04,0.207656,0.180399,0.075172,0.067242,0.206152,0.167676,0.072346,0.063188,-0.001504,-0.012723,-0.002826,-0.004054
4,sub-05,0.118592,0.128927,0.046842,0.057403,0.131206,0.125591,0.052871,0.048378,0.012614,-0.003336,0.006029,-0.009025


In [4]:
behavioral = pd.read_csv(PROCESSED_DIR / "behavioral_deltas.csv")

merged = eeg_deltas.merge(behavioral, on="participant_id", how="outer")
out_path = PROCESSED_DIR / "merged_summary.csv"
merged.to_csv(out_path, index=False)

n_eeg = eeg_deltas["participant_id"].nunique()
n_both = merged.dropna(subset=["delta_frontal_theta_variability", "delta_PVT_medianRT"]).shape[0]
print(f"Se escribió {out_path} ({len(merged)} participantes)")
print(f"Sujetos con EEG eyes-open en ambas condiciones: {n_eeg}")
print(f"Sujetos con delta de variabilidad theta Y delta de PVT (alimenta el scatter Q3): {n_both}")

Se escribió C:\Users\jimen\Downloads\data-visualization-project\deliveries\week06\data\processed\merged_summary.csv (71 participantes)
Sujetos con EEG eyes-open en ambas condiciones: 71
Sujetos con delta de variabilidad theta Y delta de PVT (alimenta el scatter Q3): 30
